# 03 — Black-Litterman

Notebooks 01 and 02 showed two ways for a mean-variance optimizer to fail:
feed it a noisy 252-day sample mean and it becomes an error-maximizer
(notebook 01); feed it a CAPM-implied mean and it algebraically collapses to
100% SPY (notebook 02, §5). Black-Litterman (BL) is the standard fix for
both: it doesn't estimate $\mu$ directly at all. It starts from a **prior**
mean implied by a reference portfolio, then **shrinks** that prior toward a
small number of explicit **views**, with the shrinkage weight set by how
confident the view is declared to be — never handing the optimizer a noisy
sample mean outright.

**The prior.** $\Pi = \delta\,\Sigma\, w_{EW}$, with $\delta = \text{sr\_ref}
/ \sigma_{EW}$ and $\text{sr\_ref}=0.3$ fixed *a priori* (a plausible
long-run diversified Sharpe, not fit to this sample). $w_{EW}$ — equal
weight — is chosen deliberately over a CAPM/SPY-implied prior: with no
views at all, BL's posterior mean collapses back to exactly $\Pi$ (shown in
§3), so an EW prior makes the **no-view portfolio the repo's own EW
benchmark** — every view is then measured against that benchmark directly. A
CAPM prior would instead make the no-view portfolio 100% SPY (§2), which
would just reproduce notebook 02's collapse and tell us nothing new about
the views themselves.

**The views.** One view per asset ($P=I$), $Q = \Pi + k\cdot
\text{sign}(\text{mom}_{12,1})\cdot\sigma$ — a **fixed, vol-scaled tilt** in
the direction of trailing 12-month-minus-1-month excess momentum, not the
trailing return itself (that would just be the sample-mean $\mu$ again, with
none of BL's shrinkage). View uncertainty follows He-Litterman:
$\Omega=\text{diag}(\text{diag}(\tau\Sigma))$, which makes $\tau$ **cancel
exactly** out of the posterior mean $\mu_{BL}$ (verified numerically in §3)
— so only $k$ and $\text{sr\_ref}$ (through $\delta$) actually matter for
the resulting weights. The optimizer is the same long-only tangency QP as
`MaxSharpe`, run on $\mu_{BL}$ against the **same** 252-day sample $\Sigma$
used by `GMV`/`MaxSharpe` (not a BL posterior covariance).

**On $k$.** The plan going in was to register $k=0.2$. A snapshot check at
2014-12-31 / 2017-12-31 / 2022-12-31 (all inside the train window) showed
that $k\ge0.2$ concentrates the portfolio hard at 2022-12-31 (effN drops
well below the 5-name floor used elsewhere in this notebook) — so $k=0.1$ is
registered instead, and $k=0.2$ is kept as a sensitivity check, not a second
"better" choice. §4 shows the mechanism directly.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
from maplab import Panel, apply_style, BlackLitterman, GMV, MaxSharpe, EqualWeight, BetaTargetMinVar, backtest, summary, tangency_closed_form

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days × {prices.shape[1]} assets "
      f"({prices.index.min().date()} → {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

In [ ]:
def ols_with_intercept(y: pd.Series, X: pd.DataFrame) -> dict:
    # OLS y ~ 1 + X via lstsq. Local helper (statsmodels is not installed
    # in this environment, so no Newey-West/HAC correction here). Returns
    # coef/tstat Series indexed ["alpha", *X.columns], R^2, and n.
    y_np = y.to_numpy()
    X_np = X.to_numpy()
    n = len(y_np)
    Xd = np.column_stack([np.ones(n), X_np])
    coefs, _, _, _ = np.linalg.lstsq(Xd, y_np, rcond=None)
    fitted = Xd @ coefs
    resid = y_np - fitted
    dof = n - Xd.shape[1]
    sigma2 = float(resid @ resid) / dof
    XtX_inv = np.linalg.inv(Xd.T @ Xd)
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    tstats = coefs / se
    ss_tot = float(np.sum((y_np - y_np.mean()) ** 2))
    ss_res = float(resid @ resid)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    names = ["alpha"] + list(X.columns)
    return {
        "coef": pd.Series(coefs, index=names),
        "tstat": pd.Series(tstats, index=names),
        "r2": r2,
        "n": n,
    }

## Why not a CAPM prior

At the last rebalance on or before the train/test split (2022-12-31), take
`Sigma` from `BlackLitterman().posterior(...)` (the same production 252-day
$\Sigma$ used everywhere else) and build a **CAPM-implied** prior instead of
the EW-implied one: $\Pi_{CAPM} = \delta\,\Sigma\, e_{SPY}$ (all weight
notionally on the market). Its unconstrained tangency portfolio is exactly
100% SPY — this is the same algebraic identity as notebook 02 §5
($\Sigma^{-1}\Pi \propto \Sigma^{-1}\Sigma e_{SPY} = e_{SPY}$ exactly, for
*any* prior proportional to $\Sigma e_{SPY}$), not a new result. It's the
reason this notebook uses an EW-implied prior instead: a CAPM prior would
make every "no-view" comparison here just re-derive notebook 02's
collapse.

In [ ]:
rebal_le_split = ml.rebalance_dates(log_returns.index)
asof_split = rebal_le_split[rebal_le_split <= split_ts].max()
print(f"snapshot asof (last rebalance <= split): {asof_split.date()}")

bl_probe = BlackLitterman()
post_probe = bl_probe.posterior(panel, asof_split)
Sigma_probe = post_probe["Sigma"]
delta_probe = post_probe["delta"]

e_spy = pd.Series(0.0, index=ml.UNIVERSE)
e_spy["SPY"] = 1.0
Pi_capm = delta_probe * (Sigma_probe.to_numpy() @ e_spy.to_numpy())
Pi_capm = pd.Series(Pi_capm, index=ml.UNIVERSE)

w_capm_tangency = tangency_closed_form(Pi_capm, Sigma_probe, 0.0)
print(f"CAPM-implied prior -> unconstrained tangency SPY weight: {w_capm_tangency['SPY']:.6f}")
print(f"max |other assets|: {w_capm_tangency.drop('SPY').abs().max():.2e}")
print("Same collapse as notebook 02 §5 — the reason this notebook uses an EW-implied prior instead.")

## Mechanics at the split snapshot ($k=0.1$)

The full posterior at the same 2022-12-31 snapshot, registered $k=0.1$: the
EW-implied prior $\Pi$, asset vol $\sigma$, trailing 12-1 excess momentum and
its sign, the view vector $Q$, and the posterior mean $\mu_{BL}$ (all
annualized, %). Then three numerical checks the design promises:

1. **$k=0$ collapses to EW exactly** (long-only weights vs $w_{EW}$).
2. **$\tau$ cancels out of $\mu_{BL}$** ($\tau=0.01$ vs $\tau=1.0$).
3. **The closed-form identity holds**: $\Sigma^{-1}\mu_{BL} = \delta\,
   w_{EW} + k\,(\Sigma+D)^{-1}(\text{sign}\cdot\sigma)$, $D=\text{diag}(\text{diag}(\Sigma))$.

Finally, the **unconstrained** BL tangency (as in notebook 01's diagnostic
style) — this is *not* what `BlackLitterman.predict_weights` returns (that's
long-only), it's a diagnostic on how hard the views pull before the
long-only constraint clips them.

In [ ]:
bl01 = BlackLitterman(k=0.1)
post = bl01.posterior(panel, asof_split)

mech_table = pd.DataFrame({
    "Pi_%": 100 * post["pi"],
    "sigma_%": 100 * post["sigma"],
    "mom12-1_%": 100 * post["momentum"],
    "sign": post["signs"],
    "Q_%": 100 * post["q"],
    "mu_BL_%": 100 * post["mu_bl"],
})
print(f"delta = {post['delta']:.4f}   #Pi<0 = {int((post['pi'] < 0).sum())} "
      f"(expected: negative Pi_i = delta*cov(r_i, r_EW) is not a bug, e.g. UUP's "
      f"negative co-movement with the rest of the universe)")
mech_table.round(3)

In [ ]:
# check 1: k=0 -> long-only weights == EW, mu_bl == pi
bl_k0 = BlackLitterman(k=0.0)
w_k0 = bl_k0(panel, asof_split)
w_ew_series = pd.Series(1.0 / len(ml.UNIVERSE), index=ml.UNIVERSE)
print(f"[k=0 -> EW] max|w_k0 - w_EW| = {(w_k0 - w_ew_series).abs().max():.2e}")

# check 2: tau invariance
mu_bl_tau_lo = BlackLitterman(k=0.1, tau=0.01).posterior(panel, asof_split)["mu_bl"]
mu_bl_tau_hi = BlackLitterman(k=0.1, tau=1.0).posterior(panel, asof_split)["mu_bl"]
print(f"[tau invariance] max|mu_BL(tau=0.01) - mu_BL(tau=1.0)| = "
      f"{(mu_bl_tau_lo - mu_bl_tau_hi).abs().max():.2e}")

# check 3: closed-form identity
Sigma_np = post["Sigma"].to_numpy()
D = np.diag(np.diag(Sigma_np))
lhs = np.linalg.solve(Sigma_np, post["mu_bl"].to_numpy())
rhs = post["delta"] * post["w_ew"].to_numpy() + 0.1 * np.linalg.solve(
    Sigma_np + D, (post["signs"] * post["sigma"]).to_numpy()
)
print(f"[closed-form identity] max|LHS - RHS| = {np.abs(lhs - rhs).max():.2e}")

# unconstrained BL tangency
w_unc = tangency_closed_form(post["mu_bl"], post["Sigma"], 0.0)
top_long = w_unc.sort_values(ascending=False).head(3)
negatives = w_unc[w_unc < 0].sort_values()
top_short = negatives.head(3)
print(f"\n[unconstrained BL(k=0.1) tangency]  gross leverage = {w_unc.abs().sum():.3f}")
print(f"  largest longs:  {dict(top_long.round(3))}")
if len(top_short):
    print(f"  largest shorts: {dict(top_short.round(3))}")
else:
    print("  largest shorts: none")

## Cautionary snapshot — 2022-12-31 (no backtest at $k=0.4$)

2022 was a near-universal risk-off year across the 13-asset universe: 11/13
assets carry a **negative** 12-1 momentum sign (table in §3) — essentially
one macro bet ("risk-off"), not 13 independent signals. The portfolio is
**long-only and fully invested** with no cash asset to retreat to, so as $k$
grows the optimizer has nowhere to put a "risk-off" view except into
whichever few assets *are* positive-momentum (DBC, UUP) — concentration, not
diversification. This cell is a snapshot only; $k=0.4$ is never backtested
(§5 onward uses $k \in \{0.1, 0.2\}$).

In [ ]:
rows = []
wtab = {}
for k in (0.1, 0.2, 0.4):
    bl_k = BlackLitterman(k=k)
    w = bl_k(panel, asof_split)
    post_k = bl_k.posterior(panel, asof_split)
    effN = 1.0 / float((w ** 2).sum())
    w_unc_k = tangency_closed_form(post_k["mu_bl"], post_k["Sigma"], 0.0)
    rows.append({
        "k": k, "effN": effN, "max_w": w.max(), "argmax": w.idxmax(),
        "unconstrained_gross_leverage": w_unc_k.abs().sum(),
    })
    wtab[f"k={k}"] = w

print(pd.DataFrame(rows).set_index("k").round(3))
print()
print("long-only weights (%):")
print((100 * pd.DataFrame(wtab)).round(1))

## Backtest comparison

Six fresh strategy instances through `ml.backtest` exactly as in notebooks
01/02: `GMV`, `MaxSharpe`, `EqualWeight`, `BetaMinVar(0.3)`, `BL(k=0.1)`
(registered), `BL(k=0.2)` (sensitivity). Same 10bps one-way cost, monthly
drift-and-rebalance, common warm-up start.

In [ ]:
strategies = {
    "GMV": GMV(),
    "MaxSharpe": MaxSharpe(),
    "EqualWeight": EqualWeight(),
    "BetaMinVar(0.3)": BetaTargetMinVar(beta_target=0.3),
    "BL(0.1)": BlackLitterman(k=0.1),
    "BL(0.2)": BlackLitterman(k=0.2),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table.round(4)

In [ ]:
bl01_strat = results["BL(0.1)"]["strat"]
bl02_strat = results["BL(0.2)"]["strat"]
for name, strat in [("BL(0.1)", bl01_strat), ("BL(0.2)", bl02_strat)]:
    print(f"{name}: fallback={len(strat.fallback_dates)}  retry={len(strat.retry_dates)}  "
          f"concentration={len(strat.concentration_dates)}")
    if strat.concentration_dates:
        print(f"  concentration dates: {[d.date() for d in strat.concentration_dates]}")

## BL(k=0.1) allocation through time

Target weights by asset group (as in notebook 02), plus effective N
($1/\sum w_i^2$) and max weight over time for `BL(0.1)` and `BL(0.2)`,
against the effN=5 concentration floor tracked in
`BlackLitterman.concentration_dates`.

In [ ]:
group_order = list(ml.ASSET_GROUPS.keys())
group_colors = plt.cm.tab10(np.linspace(0, 1, len(group_order)))

def weights_by_group(wlog):
    grouped = wlog.T.groupby(ml.GROUP_OF).sum().T
    return grouped.reindex(columns=group_order).fillna(0.0)

bl01_group = weights_by_group(results["BL(0.1)"]["wlog"])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.stackplot(bl01_group.index, bl01_group.T.to_numpy(), labels=group_order, colors=group_colors)
ax.axvline(split_ts, color="black", ls=":", lw=1)
ax.set_ylim(0, 1)
ax.set_title("BL(k=0.1) — target weights by asset group")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9.5, 6.5), sharex=True)
for name, color in [("BL(0.1)", ml.FAMILY_COLORS[BlackLitterman.family]), ("BL(0.2)", "#c0392b")]:
    wlog = results[name]["wlog"]
    effN_ts = 1.0 / (wlog ** 2).sum(axis=1)
    max_w_ts = wlog.max(axis=1)
    axes[0].plot(effN_ts.index, effN_ts.to_numpy(), label=name, color=color, lw=1.2)
    axes[1].plot(max_w_ts.index, max_w_ts.to_numpy(), label=name, color=color, lw=1.2)

axes[0].axhline(5.0, color="#888888", ls="--", lw=1, label="effN floor = 5")
axes[0].set_ylabel("effective N")
axes[0].legend(fontsize=8)
axes[0].axvline(split_ts, color="black", ls=":", lw=1)

axes[1].set_ylabel("max weight")
axes[1].axvline(split_ts, color="black", ls=":", lw=1)
axes[1].legend(fontsize=8)
axes[1].set_title("Max single-asset weight")
axes[0].set_title("Effective N")

plt.tight_layout(); plt.show()

## Views vs EW: active return

Since the no-view BL posterior is exactly EW (§3, check 1), the cleanest way
to read what the momentum views are doing is `BL(k) - EW`'s **active
return**: annualized active return, tracking error, and information ratio
(active return / tracking error), full/train/test. Cumulative active return
plotted against the train/test split.

In [ ]:
ew_net = results["EqualWeight"]["net"]

active_rows = []
active_series = {}
for name in ["BL(0.1)", "BL(0.2)"]:
    net = results[name]["net"]
    idx = net.index.intersection(ew_net.index)
    active = net.loc[idx] - ew_net.loc[idx]
    active_series[name] = active
    win_defs = {
        "full": active.index >= active.index.min(),
        "train (<= split)": active.index <= split_ts,
        "test (> split)": active.index > split_ts,
    }
    for window_name, sel in win_defs.items():
        a = active.loc[sel]
        ann_active = float(a.mean() * ml.TRADING_DAYS)
        te = float(a.std(ddof=1) * np.sqrt(ml.TRADING_DAYS))
        ir = ann_active / te if te > 0 else np.nan
        t_active = float(a.mean() / (a.std(ddof=1) / np.sqrt(len(a))))
        cost_drag_bl = float(summary_table.loc[(name, window_name), "ann_turnover"]) * ml.COST_BPS / 1e4
        cost_drag_ew = float(summary_table.loc[("EqualWeight", window_name), "ann_turnover"]) * ml.COST_BPS / 1e4
        active_gross_of_costs = ann_active + (cost_drag_bl - cost_drag_ew)
        active_rows.append({
            "strategy": name, "window": window_name,
            "ann_active_return": ann_active, "tracking_error": te,
            "information_ratio": ir, "t_active": t_active,
            "cost_drag_bl": cost_drag_bl, "cost_drag_ew": cost_drag_ew,
            "active_gross_of_costs": active_gross_of_costs,
        })

active_table = pd.DataFrame(active_rows).set_index(["strategy", "window"])
active_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.5))
for name, color in [("BL(0.1)", ml.FAMILY_COLORS[BlackLitterman.family]), ("BL(0.2)", "#c0392b")]:
    cum_active = active_series[name].cumsum()
    ax.plot(cum_active.index, cum_active.to_numpy() * 100, label=name, color=color, lw=1.3)
ax.axhline(0, color="#cccccc", lw=0.8)
ax.axvline(split_ts, color="black", ls=":", lw=1, label="train/test split")
ax.set_ylabel("cumulative active return vs EW (%, simple sum of daily active returns)")
ax.set_title("BL(k) − EW cumulative active return")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Ex-post CAPM (plain OLS)

Same regression as notebook 02 §7: realized daily **simple** excess return
(`net - rf_daily`) on the market's simple excess return (`SPY - BIL`), for
`EqualWeight`, `BL(0.1)`, `BL(0.2)`. Same caveat: no HAC correction, so
$t$-stats are indicative, not exact.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_strategy_rows = []
for name in ["EqualWeight", "BL(0.1)", "BL(0.2)"]:
    net = results[name]["net"]
    idx = net.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    strat_excess = net.loc[idx] - rf_daily.loc[idx]
    mkt_aligned = MKT_simple.loc[idx]
    fit = ols_with_intercept(strat_excess, mkt_aligned.to_frame("MKT"))
    capm_strategy_rows.append({
        "strategy": name,
        "alpha_ann": float(fit["coef"]["alpha"] * ml.TRADING_DAYS),
        "t_alpha": float(fit["tstat"]["alpha"]),
        "beta": float(fit["coef"]["MKT"]),
        "r2": fit["r2"],
        "n": fit["n"],
    })

capm_strategy_table = pd.DataFrame(capm_strategy_rows).set_index("strategy")
capm_strategy_table.round(4)

How much of the active shortfall is lower beta?

In [ ]:
alpha_ew = float(capm_strategy_table.loc["EqualWeight", "alpha_ann"])
beta_ew = float(capm_strategy_table.loc["EqualWeight", "beta"])

for name in ["BL(0.1)", "BL(0.2)"]:
    alpha_bl = float(capm_strategy_table.loc[name, "alpha_ann"])
    beta_bl = float(capm_strategy_table.loc[name, "beta"])

    net = results[name]["net"]
    idx = net.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    mkt_ann_mean = float(MKT_simple.loc[idx].mean() * ml.TRADING_DAYS)

    d_alpha = alpha_bl - alpha_ew
    d_beta = beta_bl - beta_ew
    beta_part = d_beta * mkt_ann_mean
    decomposed = d_alpha + beta_part

    realized_active_full = float(active_table.loc[(name, "full"), "ann_active_return"])
    residual = realized_active_full - decomposed

    print(f"=== {name} vs EW (full window) ===")
    print(f"  d_alpha (ann)         = {d_alpha:.4%}")
    print(f"  d_beta                = {d_beta:.4f}")
    print(f"  ann mean MKT excess   = {mkt_ann_mean:.4%}")
    print(f"  beta_part = d_beta * mkt_ann_mean = {beta_part:.4%}")
    print(f"  d_alpha + beta_part   = {decomposed:.4%}")
    print(f"  realized ann_active_return (§7, full) = {realized_active_full:.4%}")
    print(f"  residual (rf alignment/compounding, not hidden) = {residual:.4%}")
    print()

In [ ]:
def sharpe_se(sr: float, T_years: float) -> float:
    return float(np.sqrt((1 + sr ** 2 / 2) / T_years))

print("Sharpe standard error ~ sqrt((1+SR^2/2)/T), T in years (same convention as notebooks 01/02):")
for name, r in results.items():
    net_full = r["net"]
    net_test = net_full.loc[net_full.index > split_ts]
    sr_full = float(summary_table.loc[(name, "full"), "sharpe"])
    sr_test = float(summary_table.loc[(name, "test (> split)"), "sharpe"])
    T_full = len(net_full) / ml.TRADING_DAYS
    T_test = len(net_test) / ml.TRADING_DAYS
    se_full = sharpe_se(sr_full, T_full)
    se_test = sharpe_se(sr_test, T_test)
    print(f"  {name:16s} full: SR={sr_full:.2f} SE={se_full:.2f}   |   test: SR={sr_test:.2f} SE={se_test:.2f}")

**TAKEAWAY**

- **Design checks hold to floating-point precision.** A CAPM-implied prior gives an
  unconstrained tangency of exactly 100% SPY (others ≤ 2.04e-15) — notebook 02's
  collapse, so a CAPM prior would tell us nothing about the views here. k=0 reproduces EW
  (max|Δw| = 3.66e-07, the SLSQP tolerance floor), τ cancels (max diff 6.94e-18), and the
  closed-form identity holds (max error 2.97e-15).
- **Negative Π is expected.** Only UUP (Π = −1.81%) is negative at 2022-12-31: Π_i ∝
  cov(r_i, r_EW), and UUP co-moves negatively with the EW book.
- **k is a concentration knob.** At 2022-12-31, long-only effN goes 7.58 → 3.47 → 1.78 and
  unconstrained gross leverage 1.131 → 2.038 → 68.899 for k = 0.1 / 0.2 / 0.4. k was shrunk
  from the planned 0.2 to 0.1 on this train-window snapshot; 0.4 is never backtested.
- **No Sharpe difference at the ~2 SE bar.** Full window (SE ≈ 0.26–0.28): GMV 0.62,
  MaxSharpe 0.72, EW 0.74, BetaMinVar(0.3) 0.86, BL(0.1) 0.64, BL(0.2) 0.58; the widest gap
  (BetaMinVar(0.3) vs BL(0.2), 0.28) is about 1 SE. Test window (SE ≈ 0.58–0.70): widest
  gap 0.63 (MaxSharpe/BetaMinVar(0.3) at 1.13 vs BL(0.2) at 0.50), also about 1 SE.
  (Single-strategy SEs; the SE of a difference between correlated strategies is not
  computed here.)
- **The views lag EW in every window, but not significantly, and not because of costs.**
  BL(0.1) − EW: −1.36%/yr full (t = −1.78), −1.62%/yr test (t = −1.11); BL(0.2) − EW:
  −1.97%/yr full (t = −1.66), −3.01%/yr test (t = −1.33). The cost drag difference is
  only 0.23%/yr (BL(0.1), full), so gross of costs the gap is still −1.14%/yr.
- **Most of the shortfall is lower beta, not negative alpha.** Full window, BL(0.1) vs EW:
  Δβ = −0.0655, which at the realized 14.72%/yr market excess accounts for −0.96%/yr;
  Δα = −0.40%/yr. No alpha is significant (EW t = 0.49, BL(0.1) t = 0.14, BL(0.2) t = 0.32;
  plain OLS, no HAC).
- **Concentration is episodic, not only 2022.** effN < 5 on 16/208 rebalances for BL(0.1)
  and 66/208 for BL(0.2), with no fallbacks or retries. BL(0.1)'s dates cluster in
  Dec 2013 (1), Jan–Apr 2014 (4), Dec 2015 (1), Jan 2016 (1), Jul/Oct/Nov/Dec 2018 (4),
  May 2022 (1), and Apr/May/Jul/Nov 2023 (4). The 2022-12-31 snapshot shows one mechanism —
  absolute risk-off views with no cash asset force concentration into the few
  positive-momentum assets (DBC, UUP) — which motivates the Phase 3 overlay; whether the
  other episodes share that cause is not tested here.
